This notebook is designed to load neutron data and any non-neutron data (i.e. pressure, current, etc.) in an experimental folder, time bin it, and export it to CSV.

## Initialization

In [ ]:
# Importing needed code

import re
import json
from collections import defaultdict
from functools import reduce
from typing import (
    Callable,
    # TypeVar,
    # Any,
    Literal
)
from datetime import datetime, timezone, timedelta
from math import sqrt, log
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
import pandas as pd
import numpy as np
from pint import Quantity
from scipy.signal import find_peaks, peak_widths, peak_prominences

from data_processing.paths import (
    get_report_root, get_exp_root, get_reactor_data_root)
from data_processing.dataframe_validation import (
    DetectorDataframeColumn,
    BinningDataframeColumn,
    NonReactorDataframeColumn,
    SliceFitDataframeColumn
)
from data_processing.experiment_data_keys import (
    ExperimentDataKey,
    ExperimentNeutronData
)
from data_processing.loading.dataframe_loading import load_parquet_psd
from data_processing.loading.timetag_processing import (
    calculate_timetag_hours,
    calculate_event_time
)
# from data_processing.processing.bimodal_fitting import (
#     get_psd_energy_histogram,
#     scan_histogram_slices,
#     find_failed_slices,
#     BimodalBounds,
#     BimodalParams
# )
from data_processing.processing.slice_fitting import (
    get_psd_energy_histogram, scan_histogram_slices, find_failed_slices)
from data_processing.processing.calibration import Detector, recalibrate
from data_processing.processing.neutron_classification import classify
from data_processing.reporting.plotting import plot_scatter, plot_classification
from data_processing import helpers
from data_processing.processing.neutron_window_strategy.strategy_factory import \
    NeutronStrategyFactory
from data_processing.processing.neutron_window_strategy.abstract_strategy import \
    AbstractNeutronStrategy
from data_processing.types import (
    NasaGenerationSettings,
    NeutronDistributionGenerationSettings,
    NeutronWindowSettings,
    WindowType,
    SliceFitStyle,
    BimodalBounds,
    BimodalParams
)
from data_processing.loading.window_loading import (
    load_side_borders, get_neutron_window_paths)
from data_processing.loading.spectrum_unfolding import load_neutron_response_matrix
from data_processing.helpers.get_midpoints_from_bins import get_midpoints_from_bins
from data_processing.processing.spectrum_unfolding import Histogram, unfold_spectrum, strip_zeroes

In [ ]:
bins_min = helpers.get_input_with_default(
    "Enter minimum light output (in MeVee), or press Enter for default (0 MeVee)",
    0,
    float
)
bins_max = helpers.get_input_with_default(
    "Enter maximum light output (in MeVee), or press Enter for default (6 MeVee)",
    6,
    float
)
bins_width = helpers.get_input_with_default(
    "Enter light output bin width (in MeVee), or press Enter for default (0.02 MeVee)",
    0.02,
    float
)

L_bins = np.arange(bins_min, bins_max + bins_width, bins_width).tolist()

In [ ]:
R = load_neutron_response_matrix(
    Path("response_matrix_hi_res"),
    min_L=bins_min,
    max_L=bins_max,
    L_bin_widths=bins_width
)

In [ ]:
R.counts.shape

In [ ]:
np.sum(R.counts, axis=0)

In [ ]:
widths = [15, 15, 20, 15, 15, 15, 15, 15]

# test_file = Path("unfolding_test") / "output_2.45 MeV.txt"
test_file = Path("unfolding_test") / "output-AmBe.txt"
df = pd.read_fwf(test_file, widths=widths)

bins = np.arange(bins_min, bins_max + bins_width, bins_width)
cut = pd.cut(df["det_pulse (MeVee)"], bins.tolist())
cut_index = cut.cat.categories
new_df = pd.DataFrame(
    df["NPS"].groupby(cut, observed=True).sum().reindex(cut_index, fill_value=0)
)

old_index = new_df.index
if not isinstance(old_index, pd.IntervalIndex):
    raise RuntimeError(
        f"DataFrame created from cut/groupby for {test_file.name} "
        + "was not an IntervalIndex as expected"
    )
mids = old_index.mid.to_series(index=cut_index)

np_cps = new_df["NPS"].to_numpy()
np_Ls = mids.to_numpy()

N = Histogram(np_cps, np_Ls)

In [ ]:
N.counts

In [ ]:
print(R.counts.shape)
new_R, new_N, new_phi, _ = strip_zeroes(R, N, Histogram(np.ones(R.counts.shape[1]), R.y_midpoints))
print(new_R.counts.shape)
print(new_N.counts.shape)
print(np.sum(new_R.counts, axis=0))
print(new_N.counts)

In [ ]:
phi, unfold_info = unfold_spectrum(N, R, max_iterations=2000, full_info=True)
chis = unfold_info["chis"]
errors = unfold_info["errors"]

In [ ]:
len(errors)

In [ ]:
errors[-10:]

In [ ]:
phi.counts

In [ ]:
peaks, p_data = find_peaks(phi.counts, prominence=phi.counts.max() / 100)
print(phi.midpoints[peaks])

In [ ]:
figsize = (12, 10)
fig, ax = plt.subplots(figsize=figsize)
ax.plot(range(len(chis)), chis)
ax.set(yscale="log", ylabel="Chi^2/n", xlabel="Iteration", title="Stopping Criteria")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=figsize)
ax.plot(phi.midpoints, phi.counts)
# ax.plot(phi.midpoints[peaks], phi.counts[peaks], marker="x", markersize=8)
ax.vlines(x=phi.midpoints[peaks], ymin=0, ymax=phi.counts[peaks], colors="red", linestyles="dotted")
for peak_x, peak_y in zip(phi.midpoints[peaks], phi.counts[peaks]):
    ax.annotate(f"{peak_x} MeV", (peak_x, peak_y), (5, 0), textcoords="offset fontsize", arrowprops={"width": 2}, verticalalignment="center")
ax.set(xlabel="E (MeV)", ylabel="Counts", title="Unfolded Spectrum (Simulated 2.45 MeV Neutrons)")
plt.show()